# 07d · Önceden kayıtlı test: Seviye ve EMA teyitleri
## ORB + PDH/PDL, EMA 13/48/200, PMH/PML, gövde kapanışı · 1 dk veri, 2016–2026

Bu not defteri **KARARLAR.md → "Önceden kayıtlı test: Seviye ve EMA teyitleri (07d)"** bölümündeki kuralları **birebir** uygular. Kurallar 16/11/2026'da, piyasa öncesi verisi indirilmeden ve filtreler bu veride hesaplanmadan önce kayda geçirildi (commit `ddbf189`). Eşikler veriyi gördükten sonra değiştirilemez.

**Temel ORB (07c ile aynı):** aralık yönü, aralık ucuna anlık stop emri, stop aralığın karşı ucu, gün sonunda çıkış, son giriş 15:30. Aralıklar: **5 dk (ana)** ve 15 dk.

**Test edilenler (her aralıkta dört test):**
- **F1 · PDH/PDL "yol açık":** uzunda giriş > önceki seansın en yükseği, kısada giriş < önceki seansın en düşüğü
- **F2 · EMA hizalaması:** girişten önce tamamlanmış son 5 dk mumunda uzunda EMA13 > EMA48 > EMA200, kısada tersi
- **F3 · PMH/PML:** uzunda giriş > piyasa öncesi en yüksek (04:00–09:29), kısada giriş < piyasa öncesi en düşük
- **V · Gövde kapanışı ile giriş:** aralık uzunluğundaki mum aralığın dışında kapanınca, kapanış fiyatından giriş

**Doğrulandı için (her test ayrı):** (1) tek yönlü p < **0,00625** (0,025 ÷ 4, Bonferroni), (2) fark 2016–2020 ve 2021–2026'da ayrı ayrı pozitif, (3) iyimser varsayımda da pozitif, (4) filtrelerde iki grubun her birinde en az 200 işlem.

**Veri:** Alpaca 1 dk (anahtarlar Colab **Gizli anahtarlar**'dan: `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`). Seans mumları 07c'nin Drive önbelleğinden gelir; piyasa öncesi mumları ilk çalıştırmada indirilir (~20–25 dk, ücretsiz). Databento gerekmez.

> ⚠️ Öğrenme amaçlıdır, yatırım tavsiyesi değildir.

In [ ]:
#@title ⚙️ Ayarlar
ONBELLEK = "Google Drive" #@param ["Google Drive", "Colab oturumu"]
#@markdown Google Drive: 07c'de indirilen seans verisi yeniden kullanılır, piyasa öncesi verisi de oraya kaydedilir.
PERMUTASYON_TEKRAR = 5000 #@param {type:"integer"}
#@markdown ---
#@markdown **Aşağıdakiler KARARLAR.md'de sabittir — değiştirme.**
TEST_BASLANGIC = "2016-01-01"
TEST_BITIS = "2026-08-21"
ETFLER = ["SPY", "QQQ", "IWM"]
MAG7 = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]
KOMISYON_HISSE = 0.0035
KAYMA_BPS = {"ETF": 1, "Mag7": 3}
SON_GIRIS = "15:30"
ONCESI_BAS = "04:00"
ONCESI_MIN_MUM = 10
EMA_UZUNLUK = (13, 48, 200)
ALFA = 0.025 / 4            # aralık başına dört test (F1, F2, F3, V)
MIN_ISLEM = 200

## 1 · Kurulum, anahtarlar, önbellek

In [ ]:
import subprocess, sys, os, time, json
try:
    import pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def tr_sayi(x, ond=2):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    s = f"{abs(x):,.{ond}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return ("-" if x < 0 else "") + s

def tr_yuzde(x, ond=1):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    return ("-" if x < 0 else "") + "%" + tr_sayi(abs(x) * 100, ond)

def tr_tarih(d):
    return pd.Timestamp(d).strftime("%d/%m/%Y")

def dk(saat):
    s, d = saat.split(":")
    return int(s) * 60 + int(d)

def tr_tablo(t, tam=("işlem", "n", "gün", "işlem (+)", "işlem (−)"), yuzde=("isabet", "pay"), ond=3):
    b = t.copy().astype(object)
    for col in t.columns:
        for idx in t.index:
            x = t.at[idx, col]
            if isinstance(x, str):
                continue
            if col in tam:
                b.at[idx, col] = tr_sayi(x, 0)
            elif any(str(col).startswith(y) for y in yuzde):
                b.at[idx, col] = tr_yuzde(x)
            else:
                b.at[idx, col] = tr_sayi(x, ond)
    return b

from google.colab import userdata
ANAHTAR = {}
for ad in ["ALPACA_API_KEY", "ALPACA_SECRET_KEY"]:
    try:
        ANAHTAR[ad] = userdata.get(ad)
        print(f"✅ {ad} okundu")
    except Exception as e:
        raise RuntimeError(f"{ad} okunamadı. Sol menüdeki anahtar simgesinden ekle ve 'Not defteri erişimi'ni aç. ({e})")

if ONBELLEK == "Google Drive":
    from google.colab import drive
    drive.mount("/content/drive")
    KLASOR = "/content/drive/MyDrive/abd-borsa-calismalari/veri07c"
else:
    KLASOR = "/content/veri07c"
os.makedirs(KLASOR, exist_ok=True)
print("Önbellek klasörü:", KLASOR)

SEMBOLLER = ETFLER + MAG7
GRUP = {s: ("ETF" if s in ETFLER else "Mag7") for s in SEMBOLLER}
ACILIS = dk("09:30")
rng = np.random.default_rng(42)

## 2 · Veri
- **Seans (09:30–16:00):** 07c ile aynı dosyalar (`alpaca_<sembol>`); Drive'da varsa yeniden indirilmez.
- **Piyasa öncesi (04:00–09:30):** yeni dosyalar (`alpaca_oncesi_<sembol>`). Alpaca her gün için tüm uzatılmış seansı döndürür; yalnızca 04:00–09:29 tutulur.
- **META/FB:** META için 2016 başından veri gelmezse FB'den tamamlanır (07c'deki kural).

In [ ]:
ALPACA_URL = "https://data.alpaca.markets/v2/stocks/bars"

def alpaca_indir(sembol, baslangic, bitis, bolum="seans"):
    basliklar = {"APCA-API-KEY-ID": ANAHTAR["ALPACA_API_KEY"], "APCA-API-SECRET-KEY": ANAHTAR["ALPACA_SECRET_KEY"]}
    parametre = {"symbols": sembol, "timeframe": "1Min", "start": f"{baslangic}T00:00:00Z",
                 "end": f"{bitis}T23:59:00Z", "limit": 10000, "adjustment": "raw", "feed": "sip", "sort": "asc"}
    satirlar, sayfa = [], None
    while True:
        if sayfa:
            parametre["page_token"] = sayfa
        for deneme in range(5):
            y = requests.get(ALPACA_URL, headers=basliklar, params=parametre, timeout=60)
            if y.status_code == 429:
                time.sleep(5 * (deneme + 1)); continue
            y.raise_for_status(); break
        j = y.json()
        satirlar.extend(j.get("bars", {}).get(sembol, []) or [])
        sayfa = j.get("next_page_token")
        time.sleep(0.31)                      # ücretsiz plan: dakikada 200 istek
        if not sayfa:
            break
    if not satirlar:
        return pd.DataFrame()
    df = pd.DataFrame(satirlar)[["t", "o", "h", "l", "c", "v"]]
    df.index = pd.to_datetime(df.pop("t"), utc=True).dt.tz_convert("America/New_York")
    dakika = df.index.hour * 60 + df.index.minute
    if bolum == "seans":
        return df[(dakika >= ACILIS) & (dakika < dk("16:00"))].astype("float64")
    return df[(dakika >= dk(ONCESI_BAS)) & (dakika < ACILIS)].astype("float64")

def onbellekten(ad, uret):
    yol = os.path.join(KLASOR, f"{ad}.parquet")
    if os.path.exists(yol):
        return pd.read_parquet(yol)
    df = uret()
    if len(df):
        df.to_parquet(yol)
    return df

def indir_sembol(s, bolum):
    on = "alpaca_" if bolum == "seans" else "alpaca_oncesi_"
    df = onbellekten(f"{on}{s}", lambda: alpaca_indir(s, TEST_BASLANGIC, TEST_BITIS, bolum))
    if s == "META" and (df.empty or df.index.min() > pd.Timestamp("2016-02-01", tz="America/New_York")):
        bitis_fb = (df.index.min() - pd.Timedelta(days=1)).strftime("%Y-%m-%d") if len(df) else TEST_BITIS
        fb = onbellekten(f"{on}FB", lambda: alpaca_indir("FB", TEST_BASLANGIC, bitis_fb, bolum))
        df = pd.concat([fb, df]).sort_index()
        df = df[~df.index.duplicated()]
        print(f"META ({bolum}): eski dönem FB sembolünden tamamlandı.")
    return df

BAR, ONCESI = {}, {}
for s in SEMBOLLER:
    t0 = time.time()
    BAR[s] = indir_sembol(s, "seans")
    ONCESI[s] = indir_sembol(s, "oncesi")
    g1 = pd.Series(BAR[s].index.date).nunique() if len(BAR[s]) else 0
    g2 = pd.Series(ONCESI[s].index.date).nunique() if len(ONCESI[s]) else 0
    print(f"{s:5s}: seans {tr_sayi(g1, 0)} gün · piyasa öncesi {tr_sayi(g2, 0)} gün, "
          f"{tr_sayi(len(ONCESI[s]), 0)} mum ({tr_sayi(time.time() - t0, 0)} sn)")

## 3 · Veri kalite kontrolü
Piyasa öncesi işlem 2016'da ince olabilir (özellikle IWM ve bazı hisselerde). 10'dan az piyasa öncesi mumu olan günler F3 testinden çıkar (kayıttaki kural); aşağıdaki tablo bunun hangi yıllarda ne kadar olduğunu gösterir.

In [ ]:
kalite = {}
for s in SEMBOLLER:
    d = ONCESI[s]
    if d.empty:
        kalite[s] = pd.Series(dtype=float); continue
    say = pd.Series(1, index=d.index).groupby(d.index.date).size()
    yeterli = say[say >= ONCESI_MIN_MUM]
    kalite[s] = pd.Series(1, index=pd.to_datetime(yeterli.index)).groupby(lambda x: x.year).size()
kalite = pd.DataFrame(kalite).fillna(0).astype(int)
display(Markdown(f"**Yıllara göre en az {ONCESI_MIN_MUM} piyasa öncesi mumu olan gün sayısı**"))
display(kalite)

## 4 · Günlere bölme, seviyeler ve EMA
- **PDH/PDL:** aynı sembolün önceki seans gününün (SPY takvimine göre) en yükseği / en düşüğü. Önceki gün verisi yoksa F1 eksik.
- **PMH/PML:** aynı günün 04:00–09:29 mumları.
- **EMA:** seans içi 5 dk mumların kapanışı, günler arası kesintisiz; 13, 48, 200 periyot (`adjust=False`). Girişte yalnızca **tamamlanmış** 5 dk mumlar kullanılır (mumun bitişi ≤ giriş dakikası).

In [ ]:
def gunlere_bol(df):
    gunler = {}
    if df.empty:
        return gunler
    ts = df.index
    tarih = pd.to_datetime(ts.date)
    dakika = (ts.hour * 60 + ts.minute).values
    o, h, l, c = (df[k].values for k in ["o", "h", "l", "c"])
    sinir = np.flatnonzero(np.r_[True, tarih[1:] != tarih[:-1], True])
    for a, b in zip(sinir[:-1], sinir[1:]):
        if dakika[a] != ACILIS:
            continue
        gunler[tarih[a]] = dict(dk=dakika[a:b], o=o[a:b], h=h[a:b], l=l[a:b], c=c[a:b])
    return gunler

GUN = {s: gunlere_bol(BAR[s]) for s in SEMBOLLER}
TAKVIM = sorted(GUN["SPY"].keys())
ONCEKI = {TAKVIM[i]: TAKVIM[i - 1] for i in range(1, len(TAKVIM))}

PM = {}
for s in SEMBOLLER:
    d = ONCESI[s]
    PM[s] = {}
    if d.empty:
        continue
    g = d.groupby(pd.to_datetime(d.index.date))
    t = pd.DataFrame({"h": g["h"].max(), "l": g["l"].min(), "n": g.size()})
    for gun, r in t.iterrows():
        if r["n"] >= ONCESI_MIN_MUM:
            PM[s][gun] = (r["h"], r["l"])

EMA = {}
for s in SEMBOLLER:
    d = BAR[s]
    if d.empty:
        EMA[s] = {}; continue
    b5 = d["c"].resample("5min", label="left", closed="left").last().dropna()
    b5 = b5[(b5.index.hour * 60 + b5.index.minute >= ACILIS) & (b5.index.hour * 60 + b5.index.minute < dk("16:00"))]
    e = pd.DataFrame({f"e{n}": b5.ewm(span=n, adjust=False).mean() for n in EMA_UZUNLUK})
    e["sira"] = np.arange(len(e))
    e["bas"] = e.index.hour * 60 + e.index.minute
    EMA[s] = {pd.Timestamp(gun): x for gun, x in e.groupby(pd.to_datetime(e.index.date))}
print("Günler ayrıldı:", {s: len(GUN[s]) for s in SEMBOLLER})

## 5 · ORB motoru, varyant ve filtreler
Kötümser varsayım: giriş mumunda stopa değdiyse stop. İyimser: yalnızca kapanış stopun ötesindeyse. Varyantta giriş mumun kapanışında olduğu için giriş mumunda belirsizlik yoktur; sonraki mumlar iki varsayımda aynı işlenir.

In [ ]:
def or_bilgi(g, or_dk):
    dkk, o, h, l, c = g["dk"], g["o"], g["h"], g["l"], g["c"]
    k = int((dkk < ACILIS + or_dk).sum())
    if k < max(1, or_dk // 2) or k >= len(o):
        return None
    orh, orl, oro, orc = h[:k].max(), l[:k].min(), o[0], c[k - 1]
    if abs(orc - oro) < 1e-4 * oro:
        return None
    return k, orh, orl, (1 if orc > oro else -1)

def stop_cikis(g, bas, stop, yon):
    """bas indeksinden itibaren 1 dk mumlarda stop; yoksa gün sonu kapanışı."""
    h, l, o, c = g["h"], g["l"], g["o"], g["c"]
    if bas >= len(o):
        return c[-1]
    sonra = np.arange(bas, len(o))
    st = (l[sonra] <= stop) if yon == 1 else (h[sonra] >= stop)
    if st.any():
        i = sonra[np.argmax(st)]
        return min(o[i], stop) if yon == 1 else max(o[i], stop)
    return c[-1]

def orb(g, or_dk, yon_zorla=None):
    b = or_bilgi(g, or_dk)
    if b is None:
        return None
    k, orh, orl, yon = b
    if yon_zorla is not None:
        yon = yon_zorla
    dkk, o, h, l, c = g["dk"], g["o"], g["h"], g["l"], g["c"]
    seviye, stop = (orh, orl) if yon == 1 else (orl, orh)
    aday = np.arange(k, len(o))
    aday = aday[dkk[aday] <= dk(SON_GIRIS)]
    vurdu = (h[aday] >= seviye) if yon == 1 else (l[aday] <= seviye)
    if not vurdu.any():
        return None
    e = aday[np.argmax(vurdu)]
    giris = max(seviye, o[e]) if yon == 1 else min(seviye, o[e])
    risk = (giris - stop) * yon
    if risk <= 0:
        return None
    degdi = (l[e] <= stop) if yon == 1 else (h[e] >= stop)
    kesin = (c[e] <= stop) if yon == 1 else (c[e] >= stop)
    sonraki = stop_cikis(g, e + 1, stop, yon)
    return dict(yon=yon, giris=giris, stop=stop, risk=risk, giris_dk=int(dkk[e]), orh=orh, orl=orl,
                cikis_k=stop if degdi else sonraki, cikis_i=stop if kesin else sonraki)

def varyant(g, or_dk, yon, orh, orl):
    """Aralık uzunluğundaki mumlar; kapanışı aralığın dışında olan ilk mumun kapanışında giriş."""
    dkk, c = g["dk"], g["c"]
    seviye, stop = (orh, orl) if yon == 1 else (orl, orh)
    bas = ACILIS + or_dk
    while bas + or_dk <= dk(SON_GIRIS):
        m = np.flatnonzero((dkk >= bas) & (dkk < bas + or_dk))
        if len(m):
            kapanis = c[m[-1]]
            if (kapanis > seviye) if yon == 1 else (kapanis < seviye):
                risk = (kapanis - stop) * yon
                if risk <= 0:
                    return None
                return dict(giris=kapanis, risk=risk, cikis=stop_cikis(g, m[-1] + 1, stop, yon))
        bas += or_dk
    return None

def f_pdh(s, gun, r):
    dun = ONCEKI.get(gun)
    g = GUN[s].get(dun) if dun is not None else None
    if g is None:
        return np.nan
    return float(r["giris"] > g["h"].max()) if r["yon"] == 1 else float(r["giris"] < g["l"].min())

def f_pmh(s, gun, r):
    pm = PM[s].get(gun)
    if pm is None:
        return np.nan
    return float(r["giris"] > pm[0]) if r["yon"] == 1 else float(r["giris"] < pm[1])

def f_ema(s, gun, r):
    e = EMA[s].get(gun)
    if e is None:
        return np.nan
    tamam = e[e["bas"] + 5 <= r["giris_dk"]]
    if tamam.empty:
        return np.nan
    x = tamam.iloc[-1]
    if x["sira"] < max(EMA_UZUNLUK) - 1:
        return np.nan
    a, b, c = (x[f"e{n}"] for n in EMA_UZUNLUK)
    return float(a > b > c) if r["yon"] == 1 else float(a < b < c)

def net_r(giris, cikis, yon, risk, kayma):
    maliyet = 2 * KOMISYON_HISSE + kayma / 1e4 * (giris + cikis)
    return ((cikis - giris) * yon - maliyet) / risk

def islemler(or_dk):
    satir = []
    for s in SEMBOLLER:
        kayma = KAYMA_BPS[GRUP[s]]
        for gun, g in GUN[s].items():
            if gun not in ONCEKI:
                continue
            r = orb(g, or_dk)
            if r is None:
                continue
            v = varyant(g, or_dk, r["yon"], r["orh"], r["orl"])
            R_k = net_r(r["giris"], r["cikis_k"], r["yon"], r["risk"], kayma)
            R_i = net_r(r["giris"], r["cikis_i"], r["yon"], r["risk"], kayma)
            R_v = net_r(v["giris"], v["cikis"], r["yon"], v["risk"], kayma) if v else 0.0
            satir.append(dict(sembol=s, grup=GRUP[s], tarih=gun, yon=r["yon"], R_k=R_k, R_i=R_i, R_v=R_v,
                              v_islem=float(v is not None), F1=f_pdh(s, gun, r), F2=f_ema(s, gun, r),
                              F3=f_pmh(s, gun, r)))
    return pd.DataFrame(satir)

ISLEM = {}
for or_dk in [5, 15]:
    t0 = time.time()
    ISLEM[or_dk] = islemler(or_dk)
    d = ISLEM[or_dk]
    print(f"{or_dk} dk: {tr_sayi(len(d), 0)} işlem · F1 {tr_yuzde(d['F1'].mean())} sağlanıyor ({tr_sayi(d['F1'].notna().sum(), 0)} ölçülebilir) · "
          f"F2 {tr_yuzde(d['F2'].mean())} ({tr_sayi(d['F2'].notna().sum(), 0)}) · F3 {tr_yuzde(d['F3'].mean())} ({tr_sayi(d['F3'].notna().sum(), 0)}) · "
          f"varyant işlem açtı {tr_yuzde(d['v_islem'].mean())} · {tr_sayi(time.time() - t0, 0)} sn")

## 6 · Önceden kayıtlı testler
- **Filtre (F1–F3):** fark = filtre sağlanan işlemlerin ortalama net R'ı − sağlanmayanların. Gün bazında permütasyon: aynı sayıda işlemi olan günler etiket dizilerini değiş tokuş eder, gün içinde de karıştırılır (07c ile aynı).
- **Varyant (V):** her sembol-gün için fark = varyant net R (işlem yoksa 0) − temel net R. Gün bazında işaret çevirme: her günün toplam farkı rastgele ±1 ile çarpılır.

In [ ]:
def permutasyon_p(d, kol, tekrar):
    d = d.dropna(subset=[kol]).sort_values(["tarih", "sembol"])
    r = d["R_k"].values
    m = (d[kol] == 1).values
    gozlenen = r[m].mean() - r[~m].mean()
    tarih = d["tarih"].values
    sinir = np.flatnonzero(np.r_[True, tarih[1:] != tarih[:-1], True])
    gruplar = {}
    for a, b in zip(sinir[:-1], sinir[1:]):
        gruplar.setdefault(b - a, []).append(a)
    R_g, M_g = {}, {}
    for boy, baslar in gruplar.items():
        idx = np.array(baslar)[:, None] + np.arange(boy)[None, :]
        R_g[boy], M_g[boy] = r[idx], m[idx]
    toplam_r, n = r.sum(), len(r)
    say = 0
    for _ in range(tekrar):
        top_m, n_m = 0.0, 0
        for boy in R_g:
            Mp = M_g[boy][rng.permutation(len(M_g[boy]))]
            Mp = np.take_along_axis(Mp, rng.random(Mp.shape).argsort(axis=1), axis=1)
            top_m += (R_g[boy] * Mp).sum(); n_m += Mp.sum()
        if 0 < n_m < n:
            if top_m / n_m - (toplam_r - top_m) / (n - n_m) >= gozlenen:
                say += 1
    return gozlenen, (say + 1) / (tekrar + 1)

def isaret_p(d, tekrar):
    fark = (d["R_v"] - d["R_k"]).values
    gunluk = pd.Series(fark).groupby(d["tarih"].values).sum().values
    gozlenen = fark.mean()
    say = 0
    for _ in range(tekrar // 500 + 1):
        isaret = rng.choice([-1.0, 1.0], size=(500, len(gunluk)))
        say += int(((isaret @ gunluk) / len(fark) >= gozlenen).sum())
    toplam = (tekrar // 500 + 1) * 500
    return gozlenen, (say + 1) / (toplam + 1)

def filtre_fark(d, kol, R="R_k"):
    d = d.dropna(subset=[kol])
    m = d[kol] == 1
    return d.loc[m, R].mean() - d.loc[~m, R].mean() if m.any() and (~m).any() else np.nan

def varyant_fark(d, R="R_k"):
    return (d["R_v"] - d[R]).mean() if len(d) else np.nan

ADLAR = {"F1": "F1 · PDH/PDL yol açık", "F2": "F2 · EMA 13/48/200", "F3": "F3 · PMH/PML", "V": "V · gövde kapanışı"}
SONUC = []
for or_dk in [5, 15]:
    d = ISLEM[or_dk]
    yil = pd.to_datetime(d["tarih"]).dt.year
    for test in ["F1", "F2", "F3", "V"]:
        t0 = time.time()
        if test == "V":
            f_k, p = isaret_p(d, PERMUTASYON_TEKRAR)
            f_a, f_b, f_i = varyant_fark(d[yil <= 2020]), varyant_fark(d[yil >= 2021]), varyant_fark(d, "R_i")
            n_arti, n_eksi = int(d["v_islem"].sum()), len(d)
            buyukluk = True
        else:
            f_k, p = permutasyon_p(d, test, PERMUTASYON_TEKRAR)
            f_a, f_b, f_i = filtre_fark(d[yil <= 2020], test), filtre_fark(d[yil >= 2021], test), filtre_fark(d, test, "R_i")
            n_arti, n_eksi = int((d[test] == 1).sum()), int((d[test] == 0).sum())
            buyukluk = min(n_arti, n_eksi) >= MIN_ISLEM
        sartlar = [p < ALFA, f_a > 0 and f_b > 0, f_i > 0, buyukluk]
        SONUC.append({"aralık": f"{or_dk} dk", "test": ADLAR[test], "işlem (+)": n_arti, "işlem (−)": n_eksi,
                      "fark (kötümser)": f_k, "p": p, "fark 2016–2020": f_a, "fark 2021–2026": f_b,
                      "fark (iyimser)": f_i,
                      "şartlar": "".join("✅" if x else "❌" for x in sartlar),
                      "SONUÇ": "DOĞRULANDI" if all(sartlar) else "DOĞRULANMADI"})
        print(f"{or_dk} dk {test}: {time.time() - t0:.0f} sn")
SONUC = pd.DataFrame(SONUC).set_index(["aralık", "test"])
display(Markdown("### Karar tablosu\n`işlem (+)` filtre sağlanan / varyantın işlem açtığı; `işlem (−)` sağlanmayan / tüm temel işlemler. Şartlar sırasıyla: p < 0,00625 · iki dönem · iyimser · en az 200 işlem."))
display(tr_tablo(SONUC))

## 7 · Raporlanır ama karar vermez

In [ ]:
for or_dk in [5, 15]:
    d = ISLEM[or_dk]
    satir = [{"durum": "filtresiz (temel ORB)", "işlem": len(d), "net R kötümser": d["R_k"].mean(), "net R iyimser": d["R_i"].mean()}]
    for test in ["F1", "F2", "F3"]:
        for deger, ad in [(1, "sağlanan"), (0, "sağlanmayan")]:
            a = d[d[test] == deger]
            satir.append({"durum": f"{ADLAR[test]} {ad}", "işlem": len(a), "net R kötümser": a["R_k"].mean(), "net R iyimser": a["R_i"].mean()})
    ucu = d[(d["F1"] == 1) & (d["F2"] == 1) & (d["F3"] == 1)]
    satir.append({"durum": "üç filtre birlikte", "işlem": len(ucu), "net R kötümser": ucu["R_k"].mean(), "net R iyimser": ucu["R_i"].mean()})
    satir.append({"durum": "varyant (yalnız işlem açtığı günler)", "işlem": int(d["v_islem"].sum()),
                  "net R kötümser": d.loc[d["v_islem"] == 1, "R_v"].mean(), "net R iyimser": d.loc[d["v_islem"] == 1, "R_v"].mean()})
    display(Markdown(f"**{or_dk} dk · mutlak net R (işlem başı)**"))
    display(tr_tablo(pd.DataFrame(satir).set_index("durum")))

    kir = []
    for ad, alt in [("ETF", d[d["grup"] == "ETF"]), ("Mag7", d[d["grup"] == "Mag7"])] + [(s, d[d["sembol"] == s]) for s in SEMBOLLER]:
        kir.append({"kırılım": ad, "işlem": len(alt), **{t: filtre_fark(alt, t) for t in ["F1", "F2", "F3"]}, "V": varyant_fark(alt)})
    display(Markdown(f"**{or_dk} dk · fark, grup ve sembol kırılımı (kötümser)**"))
    display(tr_tablo(pd.DataFrame(kir).set_index("kırılım")))

    yil = pd.to_datetime(d["tarih"]).dt.year
    yt = pd.DataFrame({t: d.groupby(yil).apply(lambda x: filtre_fark(x, t), include_groups=False) for t in ["F1", "F2", "F3"]})
    yt["V"] = d.groupby(yil).apply(lambda x: varyant_fark(x), include_groups=False)
    yt.index.name = "yıl"
    display(Markdown(f"**{or_dk} dk · yıllara göre fark**"))
    display(tr_tablo(yt))

In [ ]:
#@title 🔗 07c vadeli skoru ile örtüşme (Drive'da 07c vadeli verisi varsa)
yol = os.path.join(KLASOR, "databento_ES_NQ_RTY.parquet")
if not os.path.exists(yol):
    print("07c vadeli verisi önbellekte yok; bu rapor atlandı (karar vermez).")
else:
    V = pd.read_parquet(yol)
    def vadeli_secimi(kok):
        sembol = V["symbol"].astype(str)
        return V[(sembol == f"{kok}.c.0") | sembol.str.match(rf"^{kok}[FGHJKMNQUVXZ]\d")]
    def vadeli_gece(kok):
        d = vadeli_secimi(kok).sort_index()
        ts = d.index.as_unit("ns").asi8
        ny = lambda gun, saat: pd.Timestamp(gun).tz_localize("America/New_York").value + int(saat * 3600e9)
        c, kimlik = d["close"].values, d["instrument_id"].values
        sonuc = {}
        for gun, dun in ONCEKI.items():
            g_bas, g_bit = np.searchsorted(ts, [ny(dun, 18), ny(gun, 9.5)])
            k_bas, k_bit = np.searchsorted(ts, [ny(dun, 9.5), ny(dun, 16)])
            if g_bit - g_bas < 60 or k_bit == k_bas or len(np.unique(np.r_[kimlik[k_bit - 1], kimlik[g_bas:g_bit]])) != 1:
                continue
            sonuc[gun] = np.sign(c[g_bit - 1] - c[k_bit - 1])
        return sonuc
    VG = {k: vadeli_gece(k) for k in ["ES", "NQ", "RTY"]}
    ESLEME = {"SPY": "ES", "QQQ": "NQ", "IWM": "RTY"}
    for or_dk in [5, 15]:
        d = ISLEM[or_dk].copy()
        d["k1"] = [float(VG[ESLEME.get(s, "NQ")].get(g, np.nan) == y) if g in VG[ESLEME.get(s, "NQ")] else np.nan
                   for s, g, y in zip(d["sembol"], d["tarih"], d["yon"])]
        print(f"{or_dk} dk · 07c şart 1 (vadeli gece getirisi işlem yönünde) ile korelasyon: "
              + ", ".join(f"{t} {tr_sayi(d[[t, 'k1']].dropna().corr().iloc[0, 1], 2)}" for t in ["F1", "F2", "F3"]))

In [ ]:
#@title 📈 Kümülatif net R: filtre sağlanan ile filtresiz (5 dk, kötümser)
d = ISLEM[5].sort_values("tarih")
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pd.to_datetime(sorted(d["tarih"].unique())), d.groupby("tarih")["R_k"].sum().cumsum().values, label="filtresiz", lw=1, color="#9aa5b1")
for test, renk in [("F1", "#2471a3"), ("F2", "#c0392b"), ("F3", "#1e8449")]:
    f = d[d[test] == 1].groupby("tarih")["R_k"].sum().cumsum()
    ax.plot(pd.to_datetime(f.index), f.values, label=ADLAR[test], lw=1.2, color=renk)
v = d.groupby("tarih")["R_v"].sum().cumsum()
ax.plot(pd.to_datetime(v.index), v.values, label=ADLAR["V"], lw=1.2, color="#7d3c98")
ax.axhline(0, color="grey", lw=0.6); ax.set_title("5 dk ORB · kümülatif net R"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8 · Kayıt ve X gönderisi
Karar satırları KARARLAR.md'deki kayıtlı bölümün altına **değiştirilmeden** eklenecek.

In [ ]:
satirlar = []
for (ar, test), r in SONUC.iterrows():
    satirlar.append(f"{ar} · {test}: {r['SONUÇ']} · fark {tr_sayi(r['fark (kötümser)'], 3)} R, p={tr_sayi(r['p'], 4)}, "
                    f"2016–2020 {tr_sayi(r['fark 2016–2020'], 3)} / 2021–2026 {tr_sayi(r['fark 2021–2026'], 3)}, "
                    f"iyimser {tr_sayi(r['fark (iyimser)'], 3)}, işlem {tr_sayi(r['işlem (+)'], 0)} / {tr_sayi(r['işlem (−)'], 0)}")
print("KARARLAR İÇİN SONUÇ SATIRLARI\n" + "-" * 40)
print("\n".join(satirlar))

kisa = {"F1 · PDH/PDL yol açık": "PDH/PDL", "F2 · EMA 13/48/200": "EMA 13/48/200", "F3 · PMH/PML": "PMH/PML", "V · gövde kapanışı": "Gövde kapanışı"}
dogru = [(ar, t) for (ar, t), r in SONUC.iterrows() if r["SONUÇ"] == "DOĞRULANDI"]
gonderi = (
    "Önceden kayıtlı test: ORB + seviye ve EMA teyitleri\n"
    "$SPY $QQQ $IWM + Mag7 · 2016–2026 · 1 dk veri\n\n"
    "Kuralları veriyi görmeden yazdım. 5 dk ORB:\n"
    + "\n".join(f"{kisa[t]}: {'doğrulandı' if r['SONUÇ'] == 'DOĞRULANDI' else 'doğrulanmadı'} ({tr_sayi(r['fark (kötümser)'], 2)} R)"
                for (ar, t), r in SONUC.iterrows() if ar == "5 dk")
    + "\n\nÖğrenme amaçlıdır, yatırım tavsiyesi değildir."
)
print("\nKOPYALAMAK İÇİN DÜZ METİN\n" + "-" * 40)
print(gonderi)
print("-" * 40, f"\n{len(gonderi)} karakter")